In [ ]:
import os, sys, subprocess, tempfile
from coffea import processor
import coffea.util
import matplotlib.pyplot as plt
import yaml

sidm_path = str(os.getcwd()).split('/sidm')[0]
if sidm_path not in sys.path:
    sys.path.insert(1, sidm_path)

from sidm.tools import utilities, scaleout, sidm_processor, llpnanoaodschema
from sidm.tools.metadata import write_run_metadata
utilities.set_plot_style()
%matplotlib inline


os.environ["X509_USER_PROXY"] = "/uscms_data/d3/murtazas/x509_proxy.pem"
scaleout.check_voms_proxy()


cluster, client = scaleout.make_lpc_client(
    min_workers=10,
    max_workers=100,
    memory='4GB',
    disk='4GB',
    scheduler_options={"dashboard_address": ":8792"}
)
print('dashboard:', cluster.dashboard_link)
client.wait_for_workers(1, timeout=600)
print('first worker connected; cluster:', cluster)

runner = processor.Runner(
    executor=processor.DaskExecutor(client=client, status=False, retries=3),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    skipbadfiles=True,
    chunksize=10000,
)


In [ ]:
# One pass per sample, saving per-sample outputs that are skipped on restart, so a
# corrupt input file (see signal_skip.json) costs one sample rather than the whole grid.
# Sum the outputs with utilities.sum_hist. The cosmic_veto collection carries
# mu_lj_vtx_chi2(_log), whose cumulative gives the per-LJ efficiency at any threshold.
channel_names = [
    "base_ljDisplacementIso",
    "base_ljDisplacementIso_cosAlpha",
    "base_ljDisplacementIso_cosAlpha_ljSpread",
    "base_ljDisplacementIso_cosAlpha_vtxChi2",
]
run_tag = "vtxChi2_signal_2018"
SKIP_JSON = "/uscms_data/d3/murtazas/runlogs/signal_skip.json"
import re, json

hist_collections = ["cosmic_veto"]

p = sidm_processor.SidmProcessor(
    channel_names,
    hist_collections,
)

max_files_signal = -1
dir_path = f"{sidm_path}/RunOutputFiles/{run_tag}"  # outside sidm/ so UploadDirectory does not ship outputs to workers
os.makedirs(dir_path, exist_ok=True)
REDIR   = "root://cmseos.fnal.gov"
EOS_DIR = f"/store/group/lpcmetx/SIDM/coffea_outputs/{os.environ['USER']}/{run_tag}"
subprocess.run(["xrdfs", REDIR, "mkdir", "-p", EOS_DIR], check=True)

for yaml_name in ("signal_4mu_v10.yaml", "signal_2mu2e_v10.yaml"):
    with open(f"../../configs/ntuples/{yaml_name}", "r") as file:
        signals_all = list(yaml.safe_load(file)["llpNanoAOD_v2"]["samples"].keys())
    for x in signals_all:
        out_file = f"{dir_path}/{x}.coffea"
        if os.path.exists(out_file):
            continue
        print(x)
        fileset = utilities.make_fileset([x], "llpNanoAOD_v2", max_files=max_files_signal,
                                         location_cfg=yaml_name, replace_xcache=True,
                                         census_skip=SKIP_JSON)
        output_signal = None
        for attempt in range(4):
            try:
                output_signal = runner.run(fileset, treename="Events", processor_instance=p)
                break
            except Exception as e:  # corrupt input file (lzma error etc.): veto it and retry the sample
                m = re.search(r"filename=\'([^\']+)\'", str(e))
                if m is None:
                    print(f"{x}: non-file failure, skipping sample: {str(e)[:200]}")
                    break
                bad = m.group(1).rsplit("/", 1)[-1]
                skip = json.load(open(SKIP_JSON))
                skip["skip"].setdefault(x, []).append(bad)
                json.dump(skip, open(SKIP_JSON, "w"), indent=2)
                print(f"{x}: vetoed corrupt file {bad}, retrying")
                fileset = utilities.make_fileset([x], "llpNanoAOD_v2", max_files=max_files_signal,
                                                 location_cfg=yaml_name, replace_xcache=True,
                                                 census_skip=SKIP_JSON)
        if output_signal is None:
            continue
        coffea.util.save(output_signal, out_file)
        # Provenance sidecar: selections, hist collections, input file list, per-sample
        # cross section, SIDM commit and coffea version, next to the .coffea it describes.
        meta_local = write_run_metadata(
            out_file,
            fileset=fileset,
            selections=channel_names,
            hist_collections=hist_collections,
            schema="LLPNanoAODSchema",
            chunksize=10000,
            extra={"run_tag": run_tag},
        )
        for local, remote in ((out_file, f"{x}.coffea"),
                              (meta_local, f"{x}.meta.yaml")):
            if subprocess.run(["xrdcp", "-f", local,
                               f"{REDIR}/{EOS_DIR}/{remote}"]).returncode != 0:
                print(f"warning: EOS copy failed for {remote} (local output kept)")


In [3]:
# (the cosAlpha / ljSpread / vtxChi2 selections all run in the single pass above)


In [4]:
client.close()

cluster.close()